# Deploy a Managed Online Endpoint

Generate the scoring function and request, create a Microsoft Entra-authenticated endpoint, deploy the registered foundation model at zero traffic, invoke it directly, and promote the tested deployment to 100% traffic.

**Source:** Adapted from [Azure/azureml-examples simple managed deployment](https://github.com/Azure/azureml-examples/blob/37c3572b3ceafdaaa90ee4503c920cfff899df1f/sdk/python/endpoints/online/managed/online-endpoints-simple-deployment.ipynb) and this repository's endpoint patterns, MIT License.

In [ ]:
from pathlib import Path
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential
from dotenv import dotenv_values
import requests
from azure.core.credentials import TokenCredential
import ast
import json
import textwrap
import jwt
from azure.ai.ml.constants import ManagedServiceIdentityType
from azure.ai.ml.entities import (
    CodeConfiguration,
    IdentityConfiguration,
    ManagedIdentityConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
)
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
for parent in (search_start, *search_start.parents):
    candidate = parent if parent.name == "workspace_readiness" else parent / "notebooks/workspace_readiness"
    if (candidate / ".env.example").is_file():
        SUITE_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run from this repository or notebooks/workspace_readiness.")

SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"

env_path = SUITE_ROOT / ".env"
if not env_path.is_file():
    raise FileNotFoundError(f"Configure {env_path} with your four Azure resource values.")
env = dotenv_values(env_path, encoding="utf-8-sig", interpolate=False)
SUBSCRIPTION_ID = (env["AZURE_SUBSCRIPTION_ID"] or "").strip()
RESOURCE_GROUP = (env["AZURE_RESOURCE_GROUP"] or "").strip()
WORKSPACE_NAME = (env["AZUREML_WORKSPACE_NAME"] or "").strip()
REGISTRY_NAME = (env["AZUREML_REGISTRY_NAME"] or "").strip()

if len(env) != 4 or not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Fill in only the four workspace and registry values in .env.")
if any(value.startswith(("<", "${")) for value in (SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Replace the .env placeholders with your Azure resource values.")

credential = AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print(f"Workspace: {WORKSPACE_NAME}; registry: {REGISTRY_NAME}")

ASSET_NAMES = {
    "data": "readiness-taxi-data",
    "environment": "readiness-taxi-environment",
    "model": "readiness-taxi-model",
}

def arm_get(credential: TokenCredential, resource_id: str, api_version: str) -> dict:
    response = requests.get(
        f"https://management.azure.com{resource_id}",
        params={"api-version": api_version},
        headers={"Authorization": f"Bearer {credential.get_token('https://management.azure.com/.default').token}"},
        timeout=60,
    )
    response.raise_for_status()
    document = response.json()
    if not isinstance(document, dict):
        raise ValueError(f"Expected an ARM resource object for {resource_id}.")
    return document

def resolve_endpoint_identity(
    credential: TokenCredential,
    subscription_id: str,
    resource_group: str,
    identity_name: str,
) -> str:
    resource_id = (
        f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
        f"/providers/Microsoft.ManagedIdentity/userAssignedIdentities/{identity_name}"
    )
    resource = arm_get(credential, resource_id, "2023-01-31")
    if resource.get("id", "").lower() != resource_id.lower():
        raise ValueError(f"Endpoint identity readback did not match {resource_id}.")
    return resource["id"]

access_token = credential.get_token("https://management.azure.com/.default")
token_claims = jwt.decode(access_token.token, options={"verify_signature": False})
ORCHESTRATOR_OBJECT_ID = token_claims["oid"]
ENDPOINT_NAME = f"{WORKSPACE_NAME}-readiness"
MODEL_NAME = ASSET_NAMES["model"]
INSTANCE_TYPE = "Standard_DS3_v2"
PUBLIC_ACCESS = "disabled"
ENDPOINT_IDENTITY_NAME = "private-mi-online-endpoint"
IDENTITY_ID = resolve_endpoint_identity(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, ENDPOINT_IDENTITY_NAME)

DEPLOYMENT_NAME = "blue"
ENVIRONMENT_NAME = ASSET_NAMES["environment"]


In [ ]:
generated_dir = SUITE_ROOT / "outputs/generated/foundations/online"
generated_code_dir = generated_dir / "code"
generated_code_dir.mkdir(parents=True, exist_ok=True)
(generated_code_dir / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
score_path = generated_code_dir / "score.py"
request_path = generated_dir / "request.json"
score_source = r'''
import json
import os
from pathlib import Path

import pandas as pd

_model = None


def init():
    global _model
    model_root = Path(os.environ["AZUREML_MODEL_DIR"])
    matches = list(model_root.rglob("model.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one model.json, found {len(matches)}")
    _model = json.loads(matches[0].read_text(encoding="utf-8"))


def run(raw_data):
    payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes)) else raw_data
    input_data = payload.get("input_data", {})
    columns = input_data.get("columns")
    rows = input_data.get("data")
    if columns != _model["features"]:
        raise ValueError(f"Expected columns in this order: {_model['features']}")
    if not isinstance(rows, list) or not rows:
        raise ValueError("Request must contain at least one row")
    frame = pd.DataFrame(rows, columns=columns).apply(pd.to_numeric, errors="raise")
    predictions = _model["intercept"]
    for feature, coefficient in zip(
        _model["features"], _model["coefficients"], strict=True
    ):
        predictions = predictions + frame[feature] * coefficient
    return {"predictions": predictions.astype(float).tolist()}
'''
score_source = textwrap.dedent(score_source).lstrip()
ast.parse(score_source, filename=str(score_path))
score_path.write_text(score_source, encoding="utf-8")
request = {
    "input_data": {
        "columns": ["tripDistance", "passengerCount"],
        "data": [[2.5, 1], [7.0, 2]],
    }
}
request_path.write_text(json.dumps(request, indent=2) + "\n", encoding="utf-8")
print(f"Generated scoring script: {score_path}")
print(f"Generated request: {request_path}")

EXPECTED_BASE_IMAGE = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"

registered_model = ml_client.models.get(MODEL_NAME, label="latest")
registered_environment = ml_client.environments.get(
    ENVIRONMENT_NAME,
    label="latest",
)
if registered_environment.image != EXPECTED_BASE_IMAGE:
    raise ValueError(
        f"Environment {registered_environment.name}:"
        f"{registered_environment.version} uses "
        f"{registered_environment.image}, expected {EXPECTED_BASE_IMAGE}. "
        "Rerun 03_register_environment.ipynb to register a new version."
    )
print(f"Model: {registered_model.name}:{registered_model.version}")
print(f"Environment: {registered_environment.name}:{registered_environment.version}")
print(f"Base image: {registered_environment.image}")

identity = None
if IDENTITY_ID:
    identity = IdentityConfiguration(
        type=ManagedServiceIdentityType.USER_ASSIGNED,
        user_assigned_identities=[ManagedIdentityConfiguration(resource_id=IDENTITY_ID)],
    )

endpoint_definition = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Azure ML workshop foundation endpoint",
    auth_mode="aad_token",
    identity=identity,
    public_network_access=PUBLIC_ACCESS,
    tags={"validation": "workspace-readiness", "purpose": "foundations"},
)
deployment_definition = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model=registered_model,
    environment=registered_environment,
    code_configuration=CodeConfiguration(
        code=str(generated_code_dir),
        scoring_script="score.py",
    ),
    instance_type=INSTANCE_TYPE,
    instance_count=1,
)

try:
    endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
    print(f"Using existing endpoint: {endpoint.name}")
except ResourceNotFoundError:
    try:
        endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint_definition).result()
    except HttpResponseError as error:
        if error.status_code == 403 and IDENTITY_ID:
            raise PermissionError(
                "The signed-in Azure CLI identity cannot assign the configured "
                "endpoint UMI. An Azure RBAC administrator must run:\n"
                f"az role assignment create --assignee-object-id {ORCHESTRATOR_OBJECT_ID} "
                "--role 'Managed Identity Operator' "
                f"--scope '{IDENTITY_ID}'\n"
                "After role propagation, rerun Cells 2-3."
            ) from error
        raise
    print(f"Created endpoint: {endpoint.name}")

deployment = ml_client.online_deployments.begin_create_or_update(deployment_definition).result()
if deployment.provisioning_state != "Succeeded":
    raise RuntimeError(f"Deployment state is {deployment.provisioning_state}")

response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    request_file=str(request_path),
)
body = json.loads(response)
assert body["predictions"] == [9.2, 20.2]
print(body)

endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
endpoint.traffic = {DEPLOYMENT_NAME: 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Traffic promoted to {DEPLOYMENT_NAME}")


## Expected Result

The deployment reaches `Succeeded`, direct invocation returns `[9.2, 20.2]`, and the tested deployment receives 100% of endpoint traffic.

Next: `../02_jobs_and_pipelines/01_submit_single_step_merge.ipynb`.